# E5 — Fine-Tuned Chronos for RUL Prediction on C-MAPSS FD001

**Study:** Agentic Multi-Machine Predictive Maintenance (AMMPM) using Time-Series Foundation Models for Explainable and Trustworthy RUL Prediction.

**Purpose:** E4 asked whether a *frozen*, never-fine-tuned Chronos could say anything useful about RUL. This notebook asks the harder, more central question: does actually adapting `amazon/chronos-t5-small`'s pretrained weights to C-MAPSS's degradation dynamics — partial fine-tuning, not training from scratch — let a time-series foundation model outperform the from-scratch attention baselines (E2 Transformer: RMSE 14.62, E3 PatchTST: RMSE 14.83)?

**A note on how this notebook reports its result.** This is framed as the study's central TSFM claim, and there is real pressure for it to "win." It won't be made to win by construction. Every number below is computed by the pipeline described in this notebook and reported as-is — including if fine-tuning does not beat E2/E3. A negative result here (fine-tuning at this scale not helping, or being outperformed by lighter from-scratch models) is still a real, reportable finding for the paper; silently adjusting the setup until the number looks right would not be.

**Compute reality check.** This machine is CPU-only (`torch.cuda.is_available() == False`), 4 threads. A single fine-tuning step (batch=8, 30-cycle context, 1-step horizon) benchmarked at ~0.8s. Exhaustively windowing every valid (context, target) pair across 80 training engines x 14 sensor channels yields on the order of ~197,000 examples per epoch — at 20 epochs that's roughly 100+ hours, not feasible here. **This run uses a fixed, seeded random subsample of the candidate window pool per epoch** (see §5) rather than exhaustive enumeration — the same practical compromise the original Chronos training recipe itself makes (random window sampling, not exhaustive enumeration, even at their much larger scale). The budget is a single named constant.

**This is the scaled-up official run** (`EXAMPLES_PER_EPOCH=3000`, ~3.75x the initial 800-example pilot), following that pilot's result (RMSE 25.31, well short of E2/E3's ~14.6-14.8) and a diagnostic from its loss curve: validation loss bottomed out around epoch 8-12 and drifted back up through epoch 20 while training loss kept falling — a sign the pilot was overfitting its small fixed subset rather than needing more epochs. More/more-diverse examples per epoch, not more epochs, was the indicated next lever, which is what this run changes.

**Reproducibility contract:** global seed fixed to 42 via `src/utils/seed.py::set_global_seed`, CPU-only execution, deterministic sequential batches. One caveat learned the hard way in this study (see E3/EX_visualizations notes): CPU multi-threaded floating-point reductions can introduce small run-to-run non-determinism *if other CPU-heavy processes are competing for cores during training* — reproducibility holds under normal, uncontended execution.


## 0. Setup


In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


def _find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing configs/paths.py is found."""
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "paths.py").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the AMMPM project root (no configs/paths.py found above "
        f"{start}). Launch Jupyter from the project root or notebooks/ directory."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.paths import CHECKPOINTS_DIR, RESULTS_DIR, ensure_dirs  # noqa: E402
from src.data.cmapss_loader import get_cmapss  # noqa: E402
from src.utils.seed import set_global_seed  # noqa: E402

SEED = 42
set_global_seed(SEED)
ensure_dirs()

from chronos import ChronosPipeline  # noqa: E402

CHRONOS_CHECKPOINT_DIR = CHECKPOINTS_DIR / "chronos_finetuned"
CHRONOS_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Global seed:  {SEED}")
print(f"Device:       {device}")
print(f"CUDA available: {torch.cuda.is_available()}")


Project root: /home/bruce-wayne-2005/industrial-ai-project
Global seed:  42
Device:       cpu
CUDA available: False


## 1. Data: NASA C-MAPSS FD001, restricted to the 14 informative sensors

The 3 operational-setting columns are near-constant in FD001 (a single operating condition, per the loader's own docstring). Empirically, 7 of the 21 sensors are known-uninformative in this subset: `sensor_1, sensor_5, sensor_10, sensor_16, sensor_18, sensor_19` are exactly constant (std == 0, or float-epsilon noise around it), and `sensor_6` has a small but genuinely nonzero std (~0.0014) — still negligible next to the next-lowest *informative* sensor (~0.038, a ~27x gap) and classified as non-informative in the same literature convention. Dropping all 10 (3 op-settings + these 7 sensors) leaves the 14-sensor selection standard in the RUL literature for FD001/FD003 (Heimes 2008 and widely repeated since) — matching this experiment's "14 series per engine."

As in E4, `normalize=False`: Chronos performs its own per-window mean-scaling internally, so raw physical-unit readings are the right input, not externally min-max-normalized ones.


In [2]:
MAX_RUL = 125
DATA = get_cmapss(fd_num=1, max_rul=MAX_RUL, normalize=False)
train_df = DATA["train_df"]
test_df = DATA["test_df"]

UNINFORMATIVE_SENSORS_FD001 = ["sensor_1", "sensor_5", "sensor_6", "sensor_10", "sensor_16", "sensor_18", "sensor_19"]
uninformative_std = train_df[UNINFORMATIVE_SENSORS_FD001].std()
informative_std = train_df[[f"sensor_{i}" for i in range(1, 22) if f"sensor_{i}" not in UNINFORMATIVE_SENSORS_FD001]].std()
assert uninformative_std.max() < informative_std.min(), (
    "expected every 'uninformative' sensor's std to fall below every 'informative' sensor's std"
)

SELECTED_SENSORS = [f"sensor_{i}" for i in range(1, 22) if f"sensor_{i}" not in UNINFORMATIVE_SENSORS_FD001]
assert len(SELECTED_SENSORS) == 14

print(f"Selected sensors ({len(SELECTED_SENSORS)}): {SELECTED_SENSORS}")
print(f"Train: {train_df['unit_number'].nunique()} engines, {len(train_df)} rows")
print(f"Test:  {test_df['unit_number'].nunique()} engines, {len(test_df)} rows")


Selected sensors (14): ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']
Train: 100 engines, 20631 rows
Test:  100 engines, 13096 rows


## 2. Train/validation engine split (for fine-tuning, not for the linear head)

Same convention as E1-E3: split the 100 **training** engines 80/20 (`random_state=SEED`) so fine-tuning can be monitored on held-out engines and the best checkpoint selected by validation loss. This is a different split from step 4's calibration/evaluation split of the **test** engines used later for the linear head — the two 80/20-shaped splits serve different purposes and must not be confused.


In [3]:
SEQUENCE_LENGTH = 30  # context cycles, matching E1-E4's window convention
PREDICTION_LENGTH = 1  # forecast horizon: next cycle

all_train_units = sorted(train_df["unit_number"].unique())
finetune_train_units, finetune_val_units = train_test_split(
    all_train_units, test_size=0.2, random_state=SEED, shuffle=True
)
print(f"Fine-tuning train engines: {len(finetune_train_units)}")
print(f"Fine-tuning val engines:   {len(finetune_val_units)}")


Fine-tuning train engines: 80
Fine-tuning val engines:   20


## 3. Build the candidate window pool

For every (engine, sensor channel) pair in each split, slide a 30-cycle context / 1-cycle target window across the trajectory — the same univariate framing used for E4's zero-shot forecasting, just enumerated exhaustively here as a *candidate pool* that §5 will subsample from.


In [4]:
def build_univariate_windows(df: pd.DataFrame, units, sensor_cols, window: int, horizon: int):
    """All valid (context, target) windows for the given engines, across all sensor_cols.

    Returns X of shape (N, window) and y of shape (N, horizon), pooling every
    engine-channel combination into one flat candidate set.
    """
    X_list, y_list = [], []
    subset = df[df["unit_number"].isin(units)]
    for _, group in subset.groupby("unit_number"):
        group = group.sort_values("time_in_cycles")
        n = len(group)
        if n < window + horizon:
            continue
        for col in sensor_cols:
            series = group[col].to_numpy(dtype=np.float32)
            for end in range(window, n - horizon + 1):
                X_list.append(series[end - window:end])
                y_list.append(series[end:end + horizon])
    return np.stack(X_list).astype(np.float32), np.stack(y_list).astype(np.float32)


X_train_pool, y_train_pool = build_univariate_windows(
    train_df, finetune_train_units, SELECTED_SENSORS, SEQUENCE_LENGTH, PREDICTION_LENGTH
)
X_val_pool, y_val_pool = build_univariate_windows(
    train_df, finetune_val_units, SELECTED_SENSORS, SEQUENCE_LENGTH, PREDICTION_LENGTH
)

print(f"Train candidate pool: {X_train_pool.shape[0]:,} windows")
print(f"Val candidate pool:   {X_val_pool.shape[0]:,} windows")


Train candidate pool: 198,254 windows
Val candidate pool:   48,580 windows


## 4. Load Chronos and freeze the first 50% of transformer layers

`amazon/chronos-t5-small` is a T5 encoder-decoder: 6 encoder layers + 6 decoder layers (12 total). "Freeze the first 50%" is interpreted as freezing the entire encoder (the first half of the encode-then-decode information flow) and training the entire decoder — a clean, unambiguous 6/12 split, rather than an arbitrary interleaving within each stack. The shared token embedding (tied to the output projection, since `tie_word_embeddings=True`) is left trainable, since it belongs to neither stack specifically and adapting it is standard practice in partial fine-tuning.


In [5]:
CHRONOS_MODEL_ID = "amazon/chronos-t5-small"

pipeline = ChronosPipeline.from_pretrained(CHRONOS_MODEL_ID, device_map="cpu", dtype=torch.float32)
model = pipeline.model.model  # the underlying T5ForConditionalGeneration
tokenizer = pipeline.tokenizer
tokenizer.config.prediction_length = PREDICTION_LENGTH  # tokenizer asserts label length == this

for p in model.encoder.parameters():
    p.requires_grad = False
for p in model.decoder.parameters():
    p.requires_grad = True
for p in model.shared.parameters():  # tied embedding / output projection
    p.requires_grad = True

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable parameters: {n_trainable:,} ({n_trainable / (n_trainable + n_frozen):.1%})")
print(f"Frozen parameters:    {n_frozen:,}")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Trainable parameters: 27,272,960 (59.1%)
Frozen parameters:    18,881,280


## 5. Pilot-scale subsample and fine-tuning configuration

`EXAMPLES_PER_EPOCH` and `VAL_EXAMPLES` are the one place this run's scope is set — a single seeded random subsample of the candidate pool, held fixed across all 20 epochs (so "epoch" keeps its usual meaning: one shuffled pass over a fixed training set, matching E1-E4's convention), rather than resampled fresh each epoch. `EXAMPLES_PER_EPOCH=3000` (up from the initial pilot's 800) is estimated at ~300s/epoch x 20 epochs ~= 1.7 hours; `VAL_EXAMPLES` is left at 200, unchanged from the pilot, since only the training budget was the diagnosed lever.


In [6]:
EXAMPLES_PER_EPOCH = 3000
VAL_EXAMPLES = 200
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
EPOCHS = 20

rng = np.random.default_rng(SEED)

train_idx = rng.choice(len(X_train_pool), size=min(EXAMPLES_PER_EPOCH, len(X_train_pool)), replace=False)
val_idx = rng.choice(len(X_val_pool), size=min(VAL_EXAMPLES, len(X_val_pool)), replace=False)

X_train, y_train = X_train_pool[train_idx], y_train_pool[train_idx]
X_val, y_val = X_val_pool[val_idx], y_val_pool[val_idx]

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_dataset = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))

shuffle_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=shuffle_generator, num_workers=0
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Sampled train set: {len(train_dataset)} windows ({len(train_loader)} batches/epoch)")
print(f"Sampled val set:   {len(val_dataset)} windows ({len(val_loader)} batches/epoch)")


Sampled train set: 3000 windows (375 batches/epoch)
Sampled val set:   200 windows (25 batches/epoch)


## 6. Fine-tuning loop

Standard Chronos training objective: `ChronosTokenizer` quantizes each real-valued window into token IDs (mean-scaled uniform binning into a 4096-token vocabulary), and the T5 model is trained with its default sequence-to-sequence cross-entropy loss (context tokens as encoder input, target tokens as decoder labels, teacher-forced). Best checkpoint (lowest validation loss) is saved to `results/checkpoints/chronos_finetuned/` via `save_pretrained`.


In [7]:
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)

train_losses, val_losses = [], []
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, n_train_examples = 0.0, 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        input_ids, attention_mask, scale = tokenizer.context_input_transform(xb)
        label_ids, _ = tokenizer.label_input_transform(yb, scale)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=label_ids)
        out.loss.backward()
        optimizer.step()
        running_loss += out.loss.item() * xb.size(0)
        n_train_examples += xb.size(0)
    train_epoch_loss = running_loss / n_train_examples

    model.eval()
    running_val_loss, n_val_examples = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            input_ids, attention_mask, scale = tokenizer.context_input_transform(xb)
            label_ids, _ = tokenizer.label_input_transform(yb, scale)
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=label_ids)
            running_val_loss += out.loss.item() * xb.size(0)
            n_val_examples += xb.size(0)
    val_epoch_loss = running_val_loss / n_val_examples

    train_losses.append(train_epoch_loss)
    val_losses.append(val_epoch_loss)

    improved = val_epoch_loss < best_val_loss
    if improved:
        best_val_loss = val_epoch_loss
        model.save_pretrained(CHRONOS_CHECKPOINT_DIR)

    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS or improved:
        marker = " *" if improved else ""
        print(f"Epoch {epoch:3d}/{EPOCHS} | train loss: {train_epoch_loss:6.3f} | val loss: {val_epoch_loss:6.3f}{marker}")

print(f"\nBest val loss: {best_val_loss:.4f} -> checkpoint saved to {CHRONOS_CHECKPOINT_DIR}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch   1/20 | train loss:  0.558 | val loss:  0.280 *


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch   2/20 | train loss:  0.250 | val loss:  0.273 *


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch   8/20 | train loss:  0.244 | val loss:  0.264 *


Epoch  10/20 | train loss:  0.242 | val loss:  0.269


Epoch  15/20 | train loss:  0.240 | val loss:  0.271


Epoch  20/20 | train loss:  0.236 | val loss:  0.266

Best val loss: 0.2637 -> checkpoint saved to /home/bruce-wayne-2005/industrial-ai-project/results/checkpoints/chronos_finetuned


## 7. Training curve


In [8]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, label="Train loss")
plt.plot(range(1, EPOCHS + 1), val_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Token-level cross-entropy loss")
plt.title(f"E5 Chronos Fine-Tune — Training/Validation Loss (C-MAPSS FD001, {EXAMPLES_PER_EPOCH} examples/epoch)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "E5_chronos_finetune_fd001_loss_curve.png", dpi=150)
plt.show()


<Figure size 800x500 with 1 Axes>

## 8. Reload the best checkpoint for inference

Training ends at epoch 20 regardless of whether that's the best epoch; reloading here guarantees inference uses the checkpoint that actually had the lowest validation loss, not just whatever weights happen to be in memory last.


In [9]:
from transformers import T5ForConditionalGeneration  # noqa: E402

best_model = T5ForConditionalGeneration.from_pretrained(CHRONOS_CHECKPOINT_DIR)
pipeline.model.model = best_model  # swap the fine-tuned weights into the existing pipeline wrapper
pipeline.model.model.eval()

print(f"Reloaded best checkpoint from {CHRONOS_CHECKPOINT_DIR}")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Reloaded best checkpoint from /home/bruce-wayne-2005/industrial-ai-project/results/checkpoints/chronos_finetuned


## 9. Inference: forecast the next cycle per sensor, for every test engine

Same mechanics as E4's zero-shot forecasting (last 30 cycles of context per sensor channel, `num_samples` sampled forecasts, mean taken as the point prediction) — but now using the fine-tuned model instead of the frozen pretrained one, and restricted to the 14 selected sensors instead of all 24 feature channels.


In [10]:
def build_last_window(df: pd.DataFrame, sensor_cols, window: int):
    X_list, y_list = [], []
    for _, group in df.groupby("unit_number"):
        group = group.sort_values("time_in_cycles")
        feats = group[sensor_cols].to_numpy(dtype=np.float32)
        rul = group["RUL"].to_numpy(dtype=np.float32)
        n = len(group)
        if n < window:
            pad = np.repeat(feats[:1], window - n, axis=0)
            feats = np.concatenate([pad, feats], axis=0)
        X_list.append(feats[-window:])
        y_list.append(rul[-1])
    return np.stack(X_list).astype(np.float32), np.array(y_list, dtype=np.float32)


X_test_context, y_test_true = build_last_window(test_df, SELECTED_SENSORS, SEQUENCE_LENGTH)
print(f"Test context windows: {X_test_context.shape}  (engines, cycles, sensors)")

NUM_SAMPLES = 20
n_test_engines = X_test_context.shape[0]
chronos_features = np.zeros((n_test_engines, len(SELECTED_SENSORS)), dtype=np.float32)

for i in tqdm(range(n_test_engines), desc="Fine-tuned Chronos forecast per engine"):
    context_batch = torch.from_numpy(X_test_context[i].T).to(torch.float32)  # (sensors, 30)
    forecast = pipeline.predict(context_batch, prediction_length=PREDICTION_LENGTH, num_samples=NUM_SAMPLES)
    chronos_features[i] = forecast.mean(dim=(1, 2)).numpy()

print(f"Chronos forecast features: {chronos_features.shape}  (engines, sensors)")


Test context windows: (100, 30, 14)  (engines, cycles, sensors)


Fine-tuned Chronos forecast per engine:   0%|          | 0/100 [00:00<?, ?it/s]

Chronos forecast features: (100, 14)  (engines, sensors)


## 10. Label-supervised linear head: 20% calibration / 80% evaluation split of test engines

Unlike E4 (fully unsupervised), this experiment's head is explicitly label-supervised: a linear model fit on the true RUL of a 20% calibration slice of the **test** engines, exactly as specified. `Ridge`, not plain `LinearRegression` — 20 calibration engines against 14 features is close enough to the underdetermined regime that plain OLS produced badly unstable, overflow-prone predictions the first time this study tried an unregularized fit on a similarly small calibration set (E4's first draft); `Ridge(alpha=1.0)` avoids repeating that failure. Predictions are clipped to `[0, max_rul]` for the same reason established there: the target is provably bounded in that range.


In [11]:
CALIB_FRACTION = 0.2
all_test_idx = np.arange(n_test_engines)
calib_idx, eval_idx = train_test_split(
    all_test_idx, test_size=1 - CALIB_FRACTION, random_state=SEED, shuffle=True
)

calib_X, calib_y = chronos_features[calib_idx], y_test_true[calib_idx]
eval_X, eval_y = chronos_features[eval_idx], y_test_true[eval_idx]

print(f"Calibration engines: {len(calib_idx)} ({CALIB_FRACTION:.0%})")
print(f"Evaluation engines:  {len(eval_idx)} ({1 - CALIB_FRACTION:.0%})")

RIDGE_ALPHA = 1.0
regression_head = Ridge(alpha=RIDGE_ALPHA)
regression_head.fit(calib_X, calib_y)

y_pred_all = np.clip(regression_head.predict(chronos_features), a_min=0.0, a_max=MAX_RUL)
eval_pred = y_pred_all[eval_idx]


Calibration engines: 20 (20%)
Evaluation engines:  80 (80%)


## 11. Evaluation: RMSE, MAE, and the PHM08 asymmetric score

**Two numbers are reported, and they answer different questions.** The spec's literal instruction is to evaluate on the 80% of test engines not used for calibration — computed below as the "official" E5 result, saved to the JSON. But E1-E4 all score on the full 100 test engines, so an 80-engine number isn't directly comparable to them (this is exactly the sample-size mismatch identified and fixed for E4's first draft — reintroduced here because the E5 spec explicitly calls for a calibration split this time). A second, supplementary all-100-engine number is also computed — using the same fitted head to predict for the calibration engines too — so the final comparison table in §14 has an apples-to-apples option available. Both are shown; neither is hidden.


In [12]:
def phm_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """PHM08 challenge asymmetric scoring function (lower is better)."""
    d = y_pred - y_true
    scores = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return float(np.sum(scores))


def compute_metrics(y_true, y_pred):
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mae = float(np.mean(np.abs(y_pred - y_true)))
    phm = phm_score(y_true, y_pred)
    return {"rmse": rmse, "mae": mae, "phm_score": phm}


official_metrics = compute_metrics(eval_y, eval_pred)
supplementary_metrics = compute_metrics(y_test_true, y_pred_all)

print("Official (spec'd: 80 evaluation engines, calibration engines excluded):")
print(f"  RMSE: {official_metrics['rmse']:.3f}  MAE: {official_metrics['mae']:.3f}  PHM: {official_metrics['phm_score']:.3f}")
print("\nSupplementary (all 100 test engines, for direct comparison with E1-E4):")
print(f"  RMSE: {supplementary_metrics['rmse']:.3f}  MAE: {supplementary_metrics['mae']:.3f}  PHM: {supplementary_metrics['phm_score']:.3f}")


Official (spec'd: 80 evaluation engines, calibration engines excluded):
  RMSE: 22.387  MAE: 17.482  PHM: 1319.195

Supplementary (all 100 test engines, for direct comparison with E1-E4):
  RMSE: 20.698  MAE: 15.886  PHM: 1355.759


## 12. Did fine-tuning beat E2/E3?

Stated plainly, before the JSON is written, so the answer isn't buried in a table later.


In [13]:
e2_path = RESULTS_DIR / "E2_transformer_fd001.json"
e3_path = RESULTS_DIR / "E3_patchtst_fd001.json"

if e2_path.exists() and e3_path.exists():
    with open(e2_path) as f:
        e2_rmse = json.load(f)["metrics"]["rmse"]
    with open(e3_path) as f:
        e3_rmse = json.load(f)["metrics"]["rmse"]

    best_prior_rmse = min(e2_rmse, e3_rmse)
    for label, rmse in [("Official (80-engine)", official_metrics["rmse"]), ("Supplementary (100-engine)", supplementary_metrics["rmse"])]:
        verdict = "BEATS" if rmse < best_prior_rmse else "does NOT beat"
        print(f"{label} RMSE {rmse:.2f} {verdict} the best from-scratch baseline (RMSE {best_prior_rmse:.2f}).")
else:
    print("E2/E3 results not found — run those notebooks first for this comparison.")


Official (80-engine) RMSE 22.39 does NOT beat the best from-scratch baseline (RMSE 14.62).
Supplementary (100-engine) RMSE 20.70 does NOT beat the best from-scratch baseline (RMSE 14.62).


## 13. Persisting results

The JSON's `metrics` field is the **official, spec'd 80-engine number** (the one comparable to how the spec defined this experiment); `supplementary_metrics_all_100_engines` is included alongside it for transparency and for the fair cross-experiment comparison, but is not the headline number. Per-engine predictions are saved for **all 100 test engines** (using the same fitted head throughout) so this file has the same shape as E1-E4's `_predictions.npz` files, for consistency with `EX_visualizations.ipynb` and any future downstream analysis.


In [14]:
metrics = {
    "experiment_id": "E5",
    "model": "chronos_finetuned",
    "dataset": "cmapss",
    "fd_subset": "FD001",
    "seed": SEED,
    "chronos_model_id": CHRONOS_MODEL_ID,
    "checkpoint_dir": str(CHRONOS_CHECKPOINT_DIR),
    "hyperparameters": {
        "sequence_length": SEQUENCE_LENGTH,
        "prediction_length": PREDICTION_LENGTH,
        "selected_sensors": SELECTED_SENSORS,
        "frozen_layers": "encoder (6 of 12 transformer layers, 50%)",
        "trainable_layers": "decoder + shared embedding",
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "examples_per_epoch": EXAMPLES_PER_EPOCH,
        "val_examples": VAL_EXAMPLES,
        "note": (
            "Official scaled-up run (examples_per_epoch=3000), following an initial "
            "800-example pilot (RMSE 25.31) whose validation loss plateaued/drifted up "
            "after ~epoch 10, indicating the training-set size (not epoch count) was the "
            "limiting factor. Still a fixed random subsample of the candidate window pool "
            "(not exhaustive windowing: the full pool is ~197,000 examples/epoch, "
            "estimated at 100+ CPU-hours) -- see notebook \u00a75 for pool sizes and rationale."
        ),
        "calib_fraction": CALIB_FRACTION,
        "eval_fraction": 1 - CALIB_FRACTION,
        "regression_head": "sklearn.Ridge",
        "ridge_alpha": RIDGE_ALPHA,
        "num_forecast_samples": NUM_SAMPLES,
        "max_rul": MAX_RUL,
    },
    "metrics": official_metrics,
    "supplementary_metrics_all_100_engines": supplementary_metrics,
    "n_calib_engines": int(len(calib_idx)),
    "n_eval_engines": int(len(eval_idx)),
    "n_test_engines_total": int(n_test_engines),
}

results_path = RESULTS_DIR / "E5_chronos_finetune_fd001.json"
with open(results_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved metrics to {results_path}")
metrics


Saved metrics to /home/bruce-wayne-2005/industrial-ai-project/results/E5_chronos_finetune_fd001.json


{'experiment_id': 'E5',
 'model': 'chronos_finetuned',
 'dataset': 'cmapss',
 'fd_subset': 'FD001',
 'seed': 42,
 'chronos_model_id': 'amazon/chronos-t5-small',
 'checkpoint_dir': '/home/bruce-wayne-2005/industrial-ai-project/results/checkpoints/chronos_finetuned',
 'hyperparameters': {'sequence_length': 30,
  'prediction_length': 1,
  'selected_sensors': ['sensor_2',
   'sensor_3',
   'sensor_4',
   'sensor_7',
   'sensor_8',
   'sensor_9',
   'sensor_11',
   'sensor_12',
   'sensor_13',
   'sensor_14',
   'sensor_15',
   'sensor_17',
   'sensor_20',
   'sensor_21'],
  'frozen_layers': 'encoder (6 of 12 transformer layers, 50%)',
  'trainable_layers': 'decoder + shared embedding',
  'epochs': 20,
  'learning_rate': 0.0001,
  'batch_size': 8,
  'examples_per_epoch': 3000,
  'val_examples': 200,
  'note': 'Official scaled-up run (examples_per_epoch=3000), following an initial 800-example pilot (RMSE 25.31) whose validation loss plateaued/drifted up after ~epoch 10, indicating the traini

In [15]:
predictions_path = RESULTS_DIR / "E5_chronos_finetune_fd001_predictions.npz"
np.savez(predictions_path, y_true=y_test_true, y_pred=y_pred_all)
print(f"Saved per-engine predictions (all {n_test_engines} test engines) to {predictions_path}")


Saved per-engine predictions (all 100 test engines) to /home/bruce-wayne-2005/industrial-ai-project/results/E5_chronos_finetune_fd001_predictions.npz


## 14. Comparison to E1-E4

For the reasons in §11, E5's row uses the **all-100-engine supplementary metrics**, not the official 80-engine ones, so this table compares like with like across all five experiments.


In [16]:
result_files = {
    "E1 (LSTM)": RESULTS_DIR / "E1_lstm_fd001.json",
    "E2 (Transformer)": RESULTS_DIR / "E2_transformer_fd001.json",
    "E3 (PatchTST)": RESULTS_DIR / "E3_patchtst_fd001.json",
    "E4 (Chronos zero-shot)": RESULTS_DIR / "E4_chronos_zeroshot_fd001.json",
}

rows = {}
for label, path in result_files.items():
    if path.exists():
        with open(path) as f:
            rows[label] = json.load(f)["metrics"]
    else:
        print(f"{label} results not found at {path} — run its notebook first for a full comparison.")

rows["E5 (Chronos fine-tuned)"] = supplementary_metrics

comparison = pd.DataFrame(rows).T
print(comparison)

best_rmse_label = comparison["rmse"].idxmin()
print(f"\nBest RMSE across all five experiments: {best_rmse_label} ({comparison.loc[best_rmse_label, 'rmse']:.2f})")


                              rmse        mae     phm_score
E1 (LSTM)                40.532047  35.096893  18182.324219
E2 (Transformer)         14.618472  10.854953    412.316498
E3 (PatchTST)            14.833669  11.972316    385.594482
E4 (Chronos zero-shot)   45.373600  39.159828  23919.400391
E5 (Chronos fine-tuned)  20.697844  15.885689   1355.759277

Best RMSE across all five experiments: E2 (Transformer) (14.62)


## Summary

This scaled-up run (3000 examples/epoch, up from the initial 800-example pilot) is the official E5 result for the paper, reported honestly regardless of whether it beats E2/E3 (see §12's explicit verdict). Results are persisted to `results/E5_chronos_finetune_fd001.json` (overwriting the pilot's numbers) and `results/E5_chronos_finetune_fd001_predictions.npz`.
